In [ ]:
import os 

os.chdir("..")

In [ ]:
from jp_imports import JPTrade
from datetime import datetime
import polars as pl

jt = JPTrade()

In [ ]:
df = jt.pull_int_jp()
df

In [ ]:
def conversion( df: pl.DataFrame) -> pl.DataFrame:

    # Precompute lowercase units and fill nulls efficiently
    df = df.with_columns(
        [
            pl.col("qty_1", "qty_2").fill_null(0),
            pl.col("unit_1").str.to_lowercase().alias("u1"),
            pl.col("unit_2").str.to_lowercase().alias("u2"),
        ]
    )

    return df.with_columns(
        qty=pl.when(pl.col("u1") == "kg")
        .then(pl.col("qty_1"))
        .when(pl.col("u2") == "kg")
        .then(pl.col("qty_2"))
        .when(pl.col("u1") == "gm")
        .then(pl.col("qty_1") / 1000)
        .when(pl.col("u2") == "gm")
        .then(pl.col("qty_2") / 1000)
        .when(pl.col("u1") == "t")
        .then(pl.col("qty_1") * 1000)
        .when(pl.col("u2") == "t")
        .then(pl.col("qty_2") * 1000)
        .when(pl.col("u1") == "l")
        .then(pl.col("qty_1") * 1)
        .when(pl.col("u2") == "l")
        .then(pl.col("qty_2") * 1)
        .when(pl.col("u1") == "doz")
        .then(pl.col("qty_1") * 0.70874)
        .when(pl.col("u2") == "doz")
        .then(pl.col("qty_2") * 0.70874)
        .when(pl.col("u1") == "m3")
        .then(pl.col("qty_1") * 353.8322)
        .when(pl.col("u2") == "m3")
        .then(pl.col("qty_2") * 353.8322)
        .when(
            (pl.col("u1") == "pfl")
            & (pl.col("hts_code").str.slice(0, 6) == "220710")
        )
        .then(pl.col("qty_1") * 0.5556)
        .when(
            (pl.col("u2") == "pfl")
            & (pl.col("hts_code").str.slice(0, 6) == "220710")
        )
        .then(pl.col("qty_1") * 0.5556)
            .when(
                (pl.col("u1") == "pfl")
                & (pl.col("hts_code").str.slice(0, 6) == "220870")
            )
            .then(pl.col("qty_1") * 2)
            .when(
                (pl.col("u2") == "pfl")
                & (pl.col("hts_code").str.slice(0, 6) == "220870")
            )
            .then(pl.col("qty_1") * 2)
            .when(pl.col("u1") == "pfl")
            .then(pl.col("qty_1") * 1.25)
            .when(pl.col("u2") == "pfl")
            .then(pl.col("qty_2") * 1.25)
            .otherwise(None),
            qtr=pl.col("date").dt.quarter(),
            fiscal_year=pl.when(pl.col("date").dt.month() > 6)
            .then(pl.col("date").dt.year() + 1)
            .otherwise(pl.col("date").dt.year()),
            month=pl.col("date").dt.month(),
            year=pl.col("date").dt.year(),
        ).drop(["u1", "u2"])
df = jt.corrections(df=df)
conversion(df=df)

In [ ]:
jt.corrections(df=df)

In [ ]:
jt.process_int_jp(time_frame="fiscal", level="hts", source="jp", corrections=True)

In [ ]:
jt.process_int_jp(time_frame="fiscal", level="hts", source="org", corrections=True)